In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update(
    {
        "text.usetex": False,
        "axes.labelsize": 20,
        "figure.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.constrained_layout.wspace": 0,
        "figure.constrained_layout.hspace": 0,
        "figure.constrained_layout.h_pad": 0,
        "figure.constrained_layout.w_pad": 0,
        "axes.linewidth": 1.2,
    }
)

import jax
import jax.numpy as jnp
from jax import flatten_util

jax.config.update("jax_enable_x64", True)

## Transfer Functions and Direct (non-QS) Fitting

This notebook shows how to:

1. convolve a stationary GP kernel with a transfer function, and
2. fit the physical parameters of a dense `ConvolvedKernel` directly to a mock light curve using `UniVarModel`.

The direct convolution is implemented in `eztaox.kernels.transfer_function.ConvolvedKernel` using the Wiener-Khinchin relation,

$$
S_{\mathrm{conv}}(f) = S_{\mathrm{base}}(f)\left|\hat\Psi(f)\right|^2,
$$

followed by an inverse FFT back to the time domain. In the final section, the likelihood uses the dense convolved kernel itself, so the optimized parameters are the physical base-kernel and transfer-function parameters rather than any surrogate-series coefficients.


In [ ]:
import numpyro
import numpyro.distributions as dist

from eztaox.fitter import random_search
from eztaox.kernels.quasisep import Exp
from eztaox.kernels.transfer_function import (
    ConvolvedKernel,
    CausalGaussianTransferFunction,
    TransferFunction,
)
from eztaox.models import UniVarModel
from eztaox.simulator import UniVarSim
from eztaox.ts_utils import add_noise

In [ ]:
class CausalTopHatTransferFunction(TransferFunction):
    """A normalized causal top-hat transfer function."""

    def evaluate(self, X1, X2):
        dt = X2 - X1 - self.shift
        half_width = self.width / 2.0
        inside = (dt >= -half_width) & (dt <= half_width)
        return jnp.where(inside, 1.0 / jnp.maximum(self.width, 1e-12), 0.0)

In [ ]:
def kernel_params(kernel):
    return {
        "scale": float(kernel.base_kernel.scale),
        "sigma": float(kernel.base_kernel.sigma),
        "width": float(kernel.transfer_function.width),
        "shift": float(kernel.transfer_function.shift),
    }

### 1. Build a convolved kernel

We start from a DRW/OU kernel and convolve it with a transfer function. The transfer function is normalized internally so that its integral is unity.

Any custom transfer function can be implemented by subclassing `TransferFunction` and defining an `evaluate(X1, X2)` method. Below we compare a built-in causal Gaussian transfer function and a custom causal top-hat transfer function.


In [ ]:
base_kernel = Exp(scale=80.0, sigma=0.2)
gaussian_transfer_function = CausalGaussianTransferFunction(width=80.0, shift=30.0)
top_hat_transfer_function = CausalTopHatTransferFunction(width=80.0, shift=30.0)

transfer_function = gaussian_transfer_function
convolved_kernel = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=transfer_function,
    n_grid=2048,
)
convolved_kernel_tophat = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=top_hat_transfer_function,
    n_grid=2048,
)

tau_grid = jnp.linspace(0.0, 300.0, 600)
psi_grid = transfer_function.evaluate(jnp.zeros_like(tau_grid), tau_grid)
psi_grid_tophat = top_hat_transfer_function.evaluate(jnp.zeros_like(tau_grid), tau_grid)
k_base = jax.vmap(lambda tau: base_kernel.evaluate(0.0, tau))(tau_grid)
k_conv = jax.vmap(lambda tau: convolved_kernel.evaluate(0.0, tau))(tau_grid)
k_conv_tophat = jax.vmap(lambda tau: convolved_kernel_tophat.evaluate(0.0, tau))(
    tau_grid
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(psi_grid),
    color="tab:purple",
    lw=2,
    label="Causal Gaussian",
)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(psi_grid_tophat),
    color="tab:green",
    lw=2,
    ls="--",
    label="Causal top-hat",
)
axes[0].set_xlabel(r"Lag $\Delta t$")
axes[0].set_ylabel(r"$\Psi(\Delta t)$")
axes[0].set_title("Transfer function")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(np.asarray(tau_grid), np.asarray(k_base), label="Base Exp kernel", lw=2)
axes[1].plot(
    np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian-convolved kernel", lw=2
)
axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(k_conv_tophat),
    label="Top-hat-convolved kernel",
    lw=2,
    ls="--",
)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel(r"$k(\Delta t)$")
axes[1].set_title("Before and after convolution")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### 2. Fit a dense `ConvolvedKernel` directly with the standard EzTaoX fitter

Below we simulate a light curve from a dense convolved kernel and then fit the physical base-kernel and transfer-function parameters using the standard **EzTaoX** `random_search` workflow. This keeps the likelihood in the dense-kernel setting while using the same fitting interface as the other tutorials.

To make the recovery visually cleaner than a single sparse random realization, this demo uses a fixed, evenly sampled mock light curve with small observational noise.

For a single-band light curve, the transfer-function **width** is much better constrained than the pure centroid **shift**, so in this demonstration we keep the shift fixed and fit the remaining positive kernel parameters.


In [ ]:
# Use a smaller FFT grid in the direct-fit demo to keep the likelihood evaluations reasonable.
true_fit_kernel = ConvolvedKernel(
    base_kernel=Exp(scale=80.0, sigma=0.2),
    transfer_function=CausalGaussianTransferFunction(width=80.0, shift=30.0),
    n_grid=512,
)
fit_kernel0 = ConvolvedKernel(
    base_kernel=Exp(scale=55.0, sigma=0.12),
    transfer_function=CausalGaussianTransferFunction(width=55.0, shift=30.0),
    n_grid=512,
)

flat_true, _ = flatten_util.ravel_pytree(true_fit_kernel)
flat0, convolved_unravel = flatten_util.ravel_pytree(fit_kernel0)

sim_params = {"log_kernel_param": jnp.log(flat_true)}
sim = UniVarSim(true_fit_kernel, 1.0, 300.0, sim_params, zero_mean=True)
sim_t = jnp.linspace(0.0, 300.0, 96)
t_obs, y_true = sim.fixed_input_fast(sim_t, jax.random.PRNGKey(12))
yerr = jnp.full_like(t_obs, 0.003)
y_obs = add_noise(y_true, yerr, jax.random.PRNGKey(13))

model = UniVarModel(t_obs, y_obs, yerr, fit_kernel0, zero_mean=True)
fit_params0 = {"log_kernel_param": jnp.log(jnp.clip(flat0, 1e-12, None))}

print("Initial log probability:", float(model.log_prob(fit_params0)))
print("True kernel parameters:", kernel_params(true_fit_kernel))
print("Initial kernel parameters:", kernel_params(fit_kernel0))

In [ ]:
def initSampler():
    log_scale = numpyro.sample("log_scale", dist.Uniform(jnp.log(30.0), jnp.log(180.0)))
    log_sigma = numpyro.sample("log_sigma", dist.Uniform(jnp.log(0.05), jnp.log(0.35)))
    log_width = numpyro.sample("log_width", dist.Uniform(jnp.log(30.0), jnp.log(180.0)))
    log_shift = jnp.log(30.0)

    log_kernel_param = jnp.stack([log_scale, log_sigma, log_width, log_shift])
    numpyro.deterministic("log_kernel_param", log_kernel_param)
    return {"log_kernel_param": log_kernel_param}

In [ ]:
fit_key = jax.random.PRNGKey(21)
nSample = 10000
nBest = 12

bestP, ll = random_search(
    model,
    initSampler,
    fit_key,
    nSample,
    nBest,
    jaxoptMethod="L-BFGS-B",
)
print("Best log probability:", float(ll))

In [ ]:
t_pred = jnp.linspace(0.0, 300.0, 400)
mu_pred, std_pred = model.pred(bestP, t_pred)
best_kernel = convolved_unravel(jnp.exp(bestP["log_kernel_param"]))

k_direct_true = jax.vmap(lambda tau: true_fit_kernel.evaluate(0.0, tau))(tau_grid)
k_direct_init = jax.vmap(lambda tau: fit_kernel0.evaluate(0.0, tau))(tau_grid)
k_direct_best = jax.vmap(lambda tau: best_kernel.evaluate(0.0, tau))(tau_grid)

kernel_rmse_init = float(jnp.sqrt(jnp.mean((k_direct_init - k_direct_true) ** 2)))
kernel_rmse_best = float(jnp.sqrt(jnp.mean((k_direct_best - k_direct_true) ** 2)))
print("Kernel RMSE (initial -> target):", kernel_rmse_init)
print("Kernel RMSE (best fit -> target):", kernel_rmse_best)
print("Recovered kernel parameters:", kernel_params(best_kernel))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].fill_between(
    np.asarray(t_pred),
    np.asarray(mu_pred - std_pred),
    np.asarray(mu_pred + std_pred),
    color="tab:blue",
    alpha=0.2,
)
axes[0].plot(
    np.asarray(t_pred),
    np.asarray(mu_pred),
    color="tab:blue",
    lw=2,
    label="random_search ConvolvedKernel fit",
)
axes[0].errorbar(
    np.asarray(t_obs),
    np.asarray(y_obs),
    np.asarray(yerr),
    fmt=".",
    color="0.2",
    label="Mock data",
)
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Flux")
axes[0].set_title("Light-curve fit")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(
    np.asarray(tau_grid), np.asarray(k_direct_true), lw=2, label="Dense target kernel"
)
axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(k_direct_init),
    lw=2,
    ls="--",
    label="Initial guess",
)
axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(k_direct_best),
    lw=2,
    ls=":",
    label="random_search best fit",
)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel(r"$k(\Delta t)$")
axes[1].set_title("Recovered covariance")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)